In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

sp500_ticker = yf.Ticker("^GSPC")
sp500 = sp500_ticker.history(period="max")

del sp500["Dividends"]
del sp500["Stock Splits"]

sp500["Tomorrow"] = sp500["Close"].shift(-1)
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)
sp500 = sp500.loc["1990-01-01":].copy()
sp500 = sp500.dropna()
sp500["Daily_Return"] = sp500["Close"].pct_change()
sp500[f"Volatility_{horizon}"] = sp500["Daily_Return"].rolling(horizon).std()
sp500[f"Volume_Ratio_{horizon}"] = sp500["Volume"] / sp500["Volume"].rolling(horizon).mean()

horizons = [2, 5, 60, 250, 1000]
new_predictors = []

for horizon in horizons:
    rolling_averages = sp500["Close"].rolling(horizon).mean()
    
    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages
    
    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500["Target"].shift(1).rolling(horizon).sum()
    
    new_predictors.extend([ratio_column, trend_column])

sp500 = sp500.dropna()

model = RandomForestClassifier(n_estimators=200, min_samples_split=100, max_depth=5, max_features="sqrt", random_state=1, n_jobs=-1)

def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict_proba(test[predictors])[:, 1]
    preds[preds >= 0.6] = 1
    preds[preds < 0.6] = 0
    
    preds = pd.Series(preds, index=test.index, name="predictions")
    combine = pd.concat([test["Target"], preds], axis=1)
    return combine

def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []
    for i in range(start, data.shape[0], step):
        print(f"Processing row {i} of {data.shape[0]}...") # Track progress
        train = data.iloc[i-2500:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    return pd.concat(all_predictions)

predictions = backtest(sp500, model, new_predictors)

print("\n=== RESULTS ===")
print("Prediction counts:")
print(predictions["predictions"].value_counts())
print(f"\nPrecision score: {precision_score(predictions['Target'], predictions['predictions']):.4f}")
print(f"Baseline (always predict up): {predictions['Target'].mean():.4f}")



Processing row 2500 of 8941...
Processing row 2750 of 8941...
Processing row 3000 of 8941...
Processing row 3250 of 8941...
Processing row 3500 of 8941...
Processing row 3750 of 8941...
Processing row 4000 of 8941...
Processing row 4250 of 8941...
Processing row 4500 of 8941...
Processing row 4750 of 8941...
Processing row 5000 of 8941...
Processing row 5250 of 8941...
Processing row 5500 of 8941...
Processing row 5750 of 8941...
Processing row 6000 of 8941...
Processing row 6250 of 8941...
Processing row 6500 of 8941...
Processing row 6750 of 8941...
Processing row 7000 of 8941...
Processing row 7250 of 8941...
Processing row 7500 of 8941...
Processing row 7750 of 8941...
Processing row 8000 of 8941...
Processing row 8250 of 8941...
Processing row 8500 of 8941...
Processing row 8750 of 8941...

=== RESULTS ===
Prediction counts:
predictions
0.0    6045
1.0     396
Name: count, dtype: int64

Precision score: 0.5455
Baseline (always predict up): 0.5398
